# הערכת פתרון OCR + Regex לזיהוי עמוד דרכון

המחברת מנתחת את שתי טבלאות התוצאות, מחלצת מתוך שמות התמונות את סוג המסמך, סוג המקרה, מספר המסמך ומספר העמוד, ומחשבת מדדים בשתי רמות:

1. **רמת מסמך** — האם זוהה לפחות עמוד חשוד אחד במסמך. מאחר שכל מסמך מכיל דרכון, המדד הוא Detection Rate / Recall ולא Accuracy.
2. **רמת עמוד** — Accuracy, Precision, Recall, F1 ומטריצת בלבול. מדדים אלה מחושבים רק לאחר מילוי מספר עמוד הדרכון האמיתי לכל מסמך.

הערה: שתי הטבלאות שסופקו נבדקות זו מול זו. אם הן זהות, הטבלה השנייה אינה Ground Truth ולכן אסור לחשב ביניהן דיוק.

In [5]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)

RUN_A_FILE = "./dataset_images_metadata.csv"
RUN_B_FILE = "./passport_detection_results.csv"

LABELS_FILE = Path("passport_page_labels.csv")
LABELS_TEMPLATE_FILE = Path("passport_page_labels_template.csv")

REQUIRED_COLUMNS = {
    "image_name",
    "matched_pattern_names",
    "verdict",
}

VERDICT_ORDER = ["NO", "MAYBE", "YES"]
VERDICT_COLORS = {
    "NO": "#D9534F",
    "MAYBE": "#F0AD4E",
    "YES": "#5CB85C",
}

PATTERN_WEIGHTS = {
    "td3_mrz_pair_strict": 2,
    "td3_mrz_upper_line": 1,
    "td3_mrz_lower_line": 1,
    "passport_word_exact": 1,
    "passport_word_ocr_tolerant": 1,
    "passport_number_label": 2,
}

## 1. טעינת שתי טבלאות התוצאות

הקוד מחפש את הקבצים קודם בתיקייה שבה נמצאת המחברת ולאחר מכן בתיקיית `upload`.

In [6]:
def resolve_input_file(file_name):
    candidates = [
        Path(file_name),
        Path("upload") / file_name,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Could not find {file_name}. Put it next to the notebook "
        "or inside an upload directory."
    )


def load_results_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")

    missing_columns = REQUIRED_COLUMNS - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing_columns)}"
        )

    if df["image_name"].duplicated().any():
        duplicates = df.loc[
            df["image_name"].duplicated(keep=False),
            "image_name",
        ].tolist()
        raise ValueError(f"Duplicate image_name values in {path.name}: {duplicates[:10]}")

    df = df.copy()
    df["matched_pattern_names"] = df["matched_pattern_names"].fillna("")
    df["verdict"] = df["verdict"].astype(str).str.upper().str.strip()

    invalid_verdicts = sorted(set(df["verdict"]) - set(VERDICT_ORDER))
    if invalid_verdicts:
        raise ValueError(f"Invalid verdict values in {path.name}: {invalid_verdicts}")

    return df


run_a_path = resolve_input_file(RUN_A_FILE)
run_b_path = resolve_input_file(RUN_B_FILE)

run_a_df = load_results_csv(run_a_path)
run_b_df = load_results_csv(run_b_path)

print(f"Run A: {run_a_path} — {len(run_a_df):,} rows")
print(f"Run B: {run_b_path} — {len(run_b_df):,} rows")

ValueError: dataset_images_metadata.csv is missing columns: ['image_name', 'matched_pattern_names', 'verdict']

## 2. בדיקת עקביות בין שתי הטבלאות

אם שתי הטבלאות זהות, נשתמש בראשונה לניתוח. זה מוכיח עקביות בין הקבצים, אבל **אינו** מוכיח דיוק מול אמת מידה.

In [ ]:
run_comparison_df = run_a_df.merge(
    run_b_df,
    on="image_name",
    how="outer",
    suffixes=("_a", "_b"),
    indicator=True,
)

run_comparison_df["same_patterns"] = (
    run_comparison_df["matched_pattern_names_a"].fillna("")
    == run_comparison_df["matched_pattern_names_b"].fillna("")
)

run_comparison_df["same_verdict"] = (
    run_comparison_df["verdict_a"].fillna("")
    == run_comparison_df["verdict_b"].fillna("")
)

different_rows_df = run_comparison_df[
    (run_comparison_df["_merge"] != "both")
    | ~run_comparison_df["same_patterns"]
    | ~run_comparison_df["same_verdict"]
].copy()

runs_are_identical = different_rows_df.empty

print(f"Runs are identical: {runs_are_identical}")
print(f"Different rows: {len(different_rows_df):,}")

if not runs_are_identical:
    display(different_rows_df.head(20))

# Run A is used as the prediction table for the remaining analysis.
predictions_df = run_a_df.copy()

## 3. חילוץ מבנה המסמך מתוך שם התמונה

דוגמה: `PDF_Edge_Case_12_page_4.jpg` הופך ל־PDF, Edge Case, מסמך 12, עמוד 4.

In [ ]:
FILE_NAME_PATTERN = (
    r"^(?P<document_type>PDF|Word)_"
    r"(?P<case_type>Edge_Case|Normal_Doc)_"
    r"(?P<case_number>\d+)_"
    r"page_(?P<page_number>\d+)"
    r"\.(?:jpg|jpeg|png)$"
)

parsed_df = predictions_df["image_name"].str.extract(
    FILE_NAME_PATTERN,
    flags=re.IGNORECASE,
)

unparsed_names = predictions_df.loc[
    parsed_df.isna().any(axis=1),
    "image_name",
]

if not unparsed_names.empty:
    raise ValueError(
        "Could not parse these image names: "
        + ", ".join(unparsed_names.head(10))
    )

parsed_df["document_type"] = (
    parsed_df["document_type"]
    .str.upper()
    .replace({"WORD": "Word"})
)

parsed_df["case_type"] = (
    parsed_df["case_type"]
    .str.lower()
    .replace({
        "edge_case": "Edge Case",
        "normal_doc": "Normal Doc",
    })
)

parsed_df["case_number"] = pd.to_numeric(
    parsed_df["case_number"],
    errors="raise",
).astype("Int64")

parsed_df["page_number"] = pd.to_numeric(
    parsed_df["page_number"],
    errors="raise",
).astype("Int64")

predictions_df = pd.concat(
    [predictions_df, parsed_df],
    axis=1,
)

predictions_df["document_id"] = (
    predictions_df["document_type"].astype(str)
    + "_"
    + predictions_df["case_type"].str.replace(" ", "_", regex=False)
    + "_"
    + predictions_df["case_number"].astype(str)
)

predictions_df[
    [
        "image_name",
        "document_id",
        "document_type",
        "case_type",
        "page_number",
        "verdict",
    ]
].head()

## 4. שחזור Score מתוך שמות התבניות

קובצי התוצאות מכילים את שמות התבניות ולא את ה־score. לכן הוא מחושב מחדש לפי המשקלים של הפתרון.

In [ ]:
def calculate_pattern_score(value):
    if pd.isna(value) or not str(value).strip():
        return 0

    names = {
        name.strip()
        for name in str(value).split(";")
        if name.strip()
    }

    unknown_patterns = names - set(PATTERN_WEIGHTS)
    if unknown_patterns:
        raise ValueError(f"Unknown patterns: {sorted(unknown_patterns)}")

    # Exact and OCR-tolerant passport words describe the same semantic evidence.
    if "passport_word_exact" in names:
        names.discard("passport_word_ocr_tolerant")

    return sum(PATTERN_WEIGHTS[name] for name in names)


def score_to_verdict(score):
    if score >= 2:
        return "YES"
    if score == 1:
        return "MAYBE"
    return "NO"


predictions_df["score"] = (
    predictions_df["matched_pattern_names"]
    .apply(calculate_pattern_score)
)

predictions_df["expected_verdict"] = (
    predictions_df["score"].apply(score_to_verdict)
)

verdict_mismatches_df = predictions_df[
    predictions_df["verdict"] != predictions_df["expected_verdict"]
]

print(f"Verdict/score mismatches: {len(verdict_mismatches_df):,}")

if not verdict_mismatches_df.empty:
    display(
        verdict_mismatches_df[
            [
                "image_name",
                "matched_pattern_names",
                "score",
                "verdict",
                "expected_verdict",
            ]
        ]
    )

## 5. סיכום ותרשימי התפלגות

כל התרשימים מוצגים בטור. הספירות ברמת עמוד מתייחסות ל־141 התמונות; הספירות ברמת מסמך מאחדות את כל העמודים השייכים לאותו מסמך.

In [ ]:
predictions_df["prediction_strict"] = (
    predictions_df["verdict"] == "YES"
).astype(int)

predictions_df["prediction_permissive"] = (
    predictions_df["verdict"].isin(["YES", "MAYBE"])
).astype(int)

document_results_df = (
    predictions_df
    .groupby(
        [
            "document_id",
            "document_type",
            "case_type",
            "case_number",
        ],
        as_index=False,
    )
    .agg(
        page_count=("page_number", "count"),
        max_score=("score", "max"),
        strict_detected=("prediction_strict", "max"),
        permissive_detected=("prediction_permissive", "max"),
        strict_flagged_pages=("prediction_strict", "sum"),
        permissive_flagged_pages=("prediction_permissive", "sum"),
    )
)

document_summary_df = pd.DataFrame({
    "metric": [
        "Images / pages",
        "Documents",
        "YES pages",
        "MAYBE pages",
        "NO pages",
        "Strict document detection rate",
        "Permissive document detection rate",
    ],
    "value": [
        len(predictions_df),
        predictions_df["document_id"].nunique(),
        (predictions_df["verdict"] == "YES").sum(),
        (predictions_df["verdict"] == "MAYBE").sum(),
        (predictions_df["verdict"] == "NO").sum(),
        document_results_df["strict_detected"].mean(),
        document_results_df["permissive_detected"].mean(),
    ],
})

document_summary_df

In [ ]:
verdict_counts = (
    predictions_df["verdict"]
    .value_counts()
    .reindex(VERDICT_ORDER, fill_value=0)
)

document_distribution = pd.crosstab(
    document_results_df["case_type"],
    document_results_df["document_type"],
).reindex(
    index=["Normal Doc", "Edge Case"],
    columns=["PDF", "Word"],
    fill_value=0,
)

detection_by_case = (
    document_results_df
    .groupby("case_type")[["strict_detected", "permissive_detected"]]
    .mean()
    .mul(100)
    .reindex(["Normal Doc", "Edge Case"])
)

pattern_counts = (
    predictions_df["matched_pattern_names"]
    .replace("", np.nan)
    .str.split(";")
    .explode()
    .dropna()
    .str.strip()
    .value_counts()
    .reindex(list(PATTERN_WEIGHTS), fill_value=0)
    .sort_values()
)

fig, axes = plt.subplots(4, 1, figsize=(12, 24))

# Plot 1: verdict distribution
bars = axes[0].bar(
    verdict_counts.index,
    verdict_counts.values,
    color=[VERDICT_COLORS[v] for v in verdict_counts.index],
)
axes[0].bar_label(bars)
axes[0].set_title("Page Verdict Distribution")
axes[0].set_xlabel("Verdict")
axes[0].set_ylabel("Number of pages")
axes[0].grid(axis="y", alpha=0.3)

# Plot 2: document distribution
document_distribution.plot(
    kind="bar",
    ax=axes[1],
    color=["#4C78A8", "#B279A2"],
)
axes[1].set_title("Document Distribution by Case and Source Type")
axes[1].set_xlabel("Case type")
axes[1].set_ylabel("Number of documents")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Document type")
axes[1].grid(axis="y", alpha=0.3)

# Plot 3: document detection rate
detection_by_case.plot(
    kind="bar",
    ax=axes[2],
    color=["#4C78A8", "#F58518"],
)
axes[2].set_title("Document Detection Rate by Case Type")
axes[2].set_xlabel("Case type")
axes[2].set_ylabel("Detection rate (%)")
axes[2].set_ylim(0, 100)
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(["YES only", "YES + MAYBE"])
axes[2].grid(axis="y", alpha=0.3)

# Plot 4: matched patterns
pattern_counts.plot(
    kind="barh",
    ax=axes[3],
    color="#54A24B",
)
axes[3].set_title("Matched Pattern Frequency")
axes[3].set_xlabel("Number of pages")
axes[3].set_ylabel("Pattern")
axes[3].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. תוצאות ברמת מסמך

מכיוון שכל מסמך מכיל עמוד דרכון אחד, `strict_detected` מציין האם היה לפחות `YES` אחד במסמך, ו־`permissive_detected` מציין האם היה לפחות `YES` או `MAYBE` אחד.

In [ ]:
document_case_metrics_df = (
    document_results_df
    .groupby(["document_type", "case_type"], as_index=False)
    .agg(
        documents=("document_id", "nunique"),
        strict_detected_documents=("strict_detected", "sum"),
        permissive_detected_documents=("permissive_detected", "sum"),
        strict_detection_rate=("strict_detected", "mean"),
        permissive_detection_rate=("permissive_detected", "mean"),
        average_strict_flagged_pages=("strict_flagged_pages", "mean"),
        average_permissive_flagged_pages=("permissive_flagged_pages", "mean"),
    )
)

document_case_metrics_df

## 7. יצירת תבנית Ground Truth ברמת עמוד

כדי לקבל Accuracy, Precision, Recall ו־F1 אמיתיים, יש למלא בטבלה את `passport_page_number` — מספר העמוד שבו נמצא הדרכון בכל מסמך. המחברת יוצרת תבנית עם שורה אחת לכל מסמך.

In [ ]:
labels_template_df = (
    predictions_df[
        [
            "document_id",
            "document_type",
            "case_type",
            "case_number",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["document_type", "case_type", "case_number"]
    )
    .reset_index(drop=True)
)

labels_template_df["passport_page_number"] = pd.NA
labels_template_df.to_csv(
    LABELS_TEMPLATE_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Saved {len(labels_template_df):,} document labels to "
    f"{LABELS_TEMPLATE_FILE}"
)
labels_template_df.head(10)

## 8. מדדי דיוק ברמת עמוד — לאחר מילוי התוויות

שנה את שם התבנית המלאה ל־`passport_page_labels.csv` והרץ את התא הבא. אם הקובץ אינו קיים, התא ידלג על החישוב במקום להמציא Ground Truth.

In [ ]:
def calculate_binary_metrics(y_true, y_pred, approach):
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        precision_score,
        recall_score,
    )

    return {
        "approach": approach,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positive_pages": int(((y_true == 0) & (y_pred == 1)).sum()),
        "missed_passport_pages": int(((y_true == 1) & (y_pred == 0)).sum()),
    }


if not LABELS_FILE.exists():
    print(
        f"{LABELS_FILE} was not found. Fill {LABELS_TEMPLATE_FILE}, "
        f"save it as {LABELS_FILE}, and run this cell again."
    )
else:
    labels_df = pd.read_csv(LABELS_FILE, encoding="utf-8-sig")
    required_label_columns = {"document_id", "passport_page_number"}
    missing_label_columns = required_label_columns - set(labels_df.columns)

    if missing_label_columns:
        raise ValueError(
            f"Missing label columns: {sorted(missing_label_columns)}"
        )

    labels_df = labels_df[["document_id", "passport_page_number"]].copy()
    labels_df["passport_page_number"] = pd.to_numeric(
        labels_df["passport_page_number"],
        errors="coerce",
    ).astype("Int64")

    if labels_df["document_id"].duplicated().any():
        raise ValueError("Each document_id must appear once in the labels file.")

    if labels_df["passport_page_number"].isna().any():
        incomplete = labels_df.loc[
            labels_df["passport_page_number"].isna(),
            "document_id",
        ].tolist()
        raise ValueError(
            "Missing passport_page_number for: "
            + ", ".join(incomplete[:10])
        )

    page_evaluation_df = predictions_df.merge(
        labels_df,
        on="document_id",
        how="left",
        validate="many_to_one",
    )

    if page_evaluation_df["passport_page_number"].isna().any():
        missing_documents = page_evaluation_df.loc[
            page_evaluation_df["passport_page_number"].isna(),
            "document_id",
        ].unique().tolist()
        raise ValueError(
            "No labels found for: "
            + ", ".join(missing_documents[:10])
        )

    page_evaluation_df["is_passport_page"] = (
        page_evaluation_df["page_number"]
        == page_evaluation_df["passport_page_number"]
    ).astype(int)

    page_metrics_df = pd.DataFrame([
        calculate_binary_metrics(
            page_evaluation_df["is_passport_page"],
            page_evaluation_df["prediction_strict"],
            "YES only",
        ),
        calculate_binary_metrics(
            page_evaluation_df["is_passport_page"],
            page_evaluation_df["prediction_permissive"],
            "YES + MAYBE",
        ),
    ])

    display(page_metrics_df)

    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

    fig, axes = plt.subplots(2, 1, figsize=(7, 12))

    for ax, column, title in [
        (axes[0], "prediction_strict", "YES only"),
        (axes[1], "prediction_permissive", "YES + MAYBE"),
    ]:
        matrix = confusion_matrix(
            page_evaluation_df["is_passport_page"],
            page_evaluation_df[column],
            labels=[0, 1],
        )
        ConfusionMatrixDisplay(
            confusion_matrix=matrix,
            display_labels=["Normal page", "Passport page"],
        ).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(f"Page-Level Confusion Matrix — {title}")

    plt.tight_layout()
    plt.show()

    top_page_df = (
        page_evaluation_df
        .sort_values(
            ["document_id", "score", "page_number"],
            ascending=[True, False, True],
        )
        .groupby("document_id", as_index=False)
        .first()
    )

    top_page_df["correct_top_page"] = (
        top_page_df["page_number"]
        == top_page_df["passport_page_number"]
    )

    top_page_df["correct_and_confident"] = (
        top_page_df["correct_top_page"]
        & (top_page_df["score"] >= 2)
    )

    localization_metrics_df = pd.DataFrame({
        "metric": [
            "Top-page accuracy",
            "Correct and confident page rate",
        ],
        "value": [
            top_page_df["correct_top_page"].mean(),
            top_page_df["correct_and_confident"].mean(),
        ],
    })

    display(localization_metrics_df)

## פירוש נכון של התוצאות

- **Strict / YES only**: Precision גבוה יותר בדרך כלל, אך עלול לפספס דרכונים.
- **Permissive / YES + MAYBE**: Recall גבוה יותר בדרך כלל, אך עלול לסמן יותר עמודים רגילים.
- **Document Detection Rate**: אחוז המסמכים שבהם נמצא לפחות עמוד חשוד. מאחר שכל המסמכים חיוביים ברמת מסמך, אי אפשר לחשב Specificity ברמת מסמך.
- **Page-level F1**: המדד המרכזי כאשר רוצים לאזן בין פספוס עמודי דרכון לבין סימון שגוי של עמודים רגילים.
- **Top-page accuracy**: בודק אם העמוד שקיבל את ה־score הגבוה ביותר הוא עמוד הדרכון האמיתי.